# DCA Buy-the-Bear v3 — Reinvestment Ratio Optimisation

Test reinvestment ratios from 1/2 to 1/20 to find the optimal balance of return vs risk.

In [ ]:
import pandas as pd
import requests
import time
import json
import numpy as np
from datetime import datetime, timezone

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: '%.6f' % x)

In [ ]:
# Fetch monthly BTCUSDT klines (same as v1/v2)

def fetch_monthly_klines(symbol='BTCUSDT', limit=500):
    url = 'https://api.binance.com/api/v3/klines'
    all_klines = []
    start_time = 0
    while True:
        params = {'symbol': symbol, 'interval': '1M', 'startTime': start_time, 'limit': limit}
        resp = requests.get(url, params=params)
        data = resp.json()
        if not data or len(data) == 0:
            break
        all_klines.extend(data)
        print(f'Fetched {len(data)} months, up to {datetime.fromtimestamp(data[-1][0]/1000, tz=timezone.utc).strftime("%Y-%m")}')
        if len(data) < limit:
            break
        start_time = data[-1][0] + 1
        time.sleep(0.1)
    return all_klines

klines = fetch_monthly_klines()

In [ ]:
# Convert to DataFrame

columns = ['timestamp', 'open', 'high', 'low', 'close', 'volume',
           'close_time', 'quote_vol', 'trades', 'taker_buy_base', 'taker_buy_quote', 'ignore']
df = pd.DataFrame(klines, columns=columns)
df['date'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
for c in ['open', 'high', 'low', 'close', 'volume']:
    df[c] = df[c].astype(float)
df = df[['date', 'open', 'high', 'low', 'close', 'volume']].copy()
df = df.sort_values('date').reset_index(drop=True)
print(f'{df["date"].min().strftime("%Y-%m")} -> {df["date"].max().strftime("%Y-%m")} ({len(df)} months)')

In [ ]:
# Run backtest for a given reinvestment ratio
# ratio = 1/N where N is the divisor (e.g., N=15 means reinvest 1/15 of reserve)

def run_backtest(divisor):
    btc_held = 0.0
    usdt_reserve = 0.0
    total_invested = 0.0
    highest_close = 0.0
    records = []

    for i, row in df.iterrows():
        close = row['close']
        month = row['date']
        action = 'nothing'

        if close < row['open']:
            extra = usdt_reserve / divisor if usdt_reserve > 0 else 0.0
            total_usdt = 10.0 + extra
            btc_bought = total_usdt / close
            btc_held += btc_bought
            total_invested += 10.0
            usdt_reserve -= extra

        prev_highest = highest_close
        if close > highest_close:
            highest_close = close

        if btc_held > 0.000001 and close > prev_highest:
            sell_btc = btc_held * 0.7
            sell_usdt = sell_btc * close
            btc_held -= sell_btc
            usdt_reserve += sell_usdt

        portfolio_value = btc_held * close + usdt_reserve
        records.append(portfolio_value)

    final_val = records[-1]
    total_pnl = final_val - total_invested
    ret_pct = (final_val / total_invested - 1) * 100

    # Max drawdown
    eq = pd.Series(records)
    mask = eq > 0
    eq_pos = eq[mask]
    running_max = eq_pos.cummax()
    dd = (running_max - eq_pos) / running_max
    max_dd = dd.max() * 100 if len(dd) > 0 else 0.0

    # Sharpe (annualized)
    monthly_returns = eq.pct_change().dropna()
    monthly_returns = monthly_returns[monthly_returns.index >= 12]  # skip warmup
    sharpe = (monthly_returns.mean() / monthly_returns.std()) * np.sqrt(12) if monthly_returns.std() > 0 else 0.0

    # Profit factor
    pos_sum = monthly_returns[monthly_returns > 0].sum()
    neg_sum = monthly_returns[monthly_returns < 0].abs().sum()
    pf = pos_sum / neg_sum if neg_sum > 0 else float('inf')

    # Calmar ratio: annualized return / max drawdown
    annualized_ret = (1 + ret_pct / 100) ** (12 / len(df)) - 1
    calmar = annualized_ret / (max_dd / 100) if max_dd > 0 else 0

    return {
        'divisor': divisor,
        'ratio': f'1/{divisor}',
        'final_val': final_val,
        'total_invested': total_invested,
        'pnl': total_pnl,
        'return_pct': ret_pct,
        'max_dd': max_dd,
        'sharpe': sharpe,
        'profit_factor': pf,
        'calmar': calmar,
        'btc_held': btc_held,
        'usdt_reserve': usdt_reserve,
    }

# Test divisors from 2 to 20
results_list = []
for d in range(2, 21):
    r = run_backtest(d)
    results_list.append(r)
    sym = 'BEST' if d in (2, 3, 4, 5, 6) else ''  # placeholder
    print(f"1/{d:2d} | return={r['return_pct']:7.2f}% | DD={r['max_dd']:5.2f}% | Sharpe={r['sharpe']:.2f} | Calmar={r['calmar']:.2f} | BTC={r['btc_held']:.6f}")

opt_results = pd.DataFrame(results_list)
print('\nDone')

In [ ]:
# Results table

pd.set_option('display.max_rows', None)
opt_results[['ratio', 'return_pct', 'max_dd', 'sharpe', 'calmar', 'profit_factor', 'pnl', 'btc_held']]

In [ ]:
# Identify optimal by different criteria

best_sharpe = opt_results.loc[opt_results['sharpe'].idxmax()]
best_calmar = opt_results.loc[opt_results['calmar'].idxmax()]
best_return = opt_results.loc[opt_results['return_pct'].idxmax()]

print("="*60)
print("OPTIMUM BY DIFFERENT METRICS")
print("="*60)
print(f"By Sharpe ratio:   1/{int(best_sharpe['divisor'])}  Sharpe={best_sharpe['sharpe']:.2f}  Return={best_sharpe['return_pct']:.1f}%  DD={best_sharpe['max_dd']:.1f}%")
print(f"By Calmar ratio:   1/{int(best_calmar['divisor'])}  Calmar={best_calmar['calmar']:.2f}  Return={best_calmar['return_pct']:.1f}%  DD={best_calmar['max_dd']:.1f}%")
print(f"By return:         1/{int(best_return['divisor'])}  Return={best_return['return_pct']:.1f}%  DD={best_return['max_dd']:.1f}%  Sharpe={best_return['sharpe']:.2f}")

In [ ]:
# Chart: metrics vs reinvestment ratio

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

labels = [f'1/{d}' for d in opt_results['divisor']]
x = range(len(labels))

# Return %
ax = axes[0, 0]
ax.plot(x, opt_results['return_pct'], 'bo-', linewidth=2, markersize=6)
best_idx = opt_results['return_pct'].idxmax()
ax.plot(best_idx, opt_results.loc[best_idx, 'return_pct'], 'r*', markersize=15, label=f'Best: {opt_results.loc[best_idx, "ratio"]}')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45)
ax.set_ylabel('Return (%)')
ax.set_title('Total Return')
ax.legend()
ax.grid(True, alpha=0.3)

# Max DD
ax = axes[0, 1]
ax.plot(x, opt_results['max_dd'], 'ro-', linewidth=2, markersize=6)
min_idx = opt_results['max_dd'].idxmin()
ax.plot(min_idx, opt_results.loc[min_idx, 'max_dd'], 'g*', markersize=15, label=f'Lowest DD: {opt_results.loc[min_idx, "ratio"]}')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45)
ax.set_ylabel('Max Drawdown (%)')
ax.set_title('Max Drawdown')
ax.legend()
ax.grid(True, alpha=0.3)

# Sharpe
ax = axes[1, 0]
ax.plot(x, opt_results['sharpe'], 'go-', linewidth=2, markersize=6)
best_idx = opt_results['sharpe'].idxmax()
ax.plot(best_idx, opt_results.loc[best_idx, 'sharpe'], 'r*', markersize=15, label=f'Best: {opt_results.loc[best_idx, "ratio"]}')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45)
ax.set_ylabel('Sharpe Ratio')
ax.set_title('Sharpe Ratio (annualized)')
ax.legend()
ax.grid(True, alpha=0.3)

# Calmar
ax = axes[1, 1]
ax.plot(x, opt_results['calmar'], 'mo-', linewidth=2, markersize=6)
best_idx = opt_results['calmar'].idxmax()
ax.plot(best_idx, opt_results.loc[best_idx, 'calmar'], 'r*', markersize=15, label=f'Best: {opt_results.loc[best_idx, "ratio"]}')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45)
ax.set_ylabel('Calmar Ratio')
ax.set_title('Calmar Ratio (ann. return / max DD)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('Reinvestment Ratio Optimisation — DCA Buy-the-Bear v3', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('dca-bt-buy-bot-optimisation.png', dpi=150, bbox_inches='tight')
print('Chart saved as dca-bt-buy-bot-optimisation.png')
plt.show()

In [ ]:
# Summary comparison: best by Sharpe, best by Calmar, vs 1/10 and 1/15

comparison = opt_results[opt_results['divisor'].isin([best_sharpe['divisor'], best_calmar['divisor'], 10, 15])].copy()
comparison['ratio'] = comparison['divisor'].apply(lambda d: f'1/{int(d)}')
comparison[['ratio', 'return_pct', 'max_dd', 'sharpe', 'calmar', 'pnl']]